# Evaluating the Role of Data Engineering

Two workflows build **the same training dataset** (hourly taxi demand per pickup zone, see `config/ml.yaml`) and train **the same model** (GBT, same parameters, same seed):

* **Approach A - from the raw files.** Everything is written in this notebook: loading the original Parquet/CSV files, cleaning and validating them, integrating them, building the zone x hour dataset and engineering the features. It uses only PySpark and the `holidays` package - no module from `src/`.
* **Approach B - from the platform.** Starts from the integrated Gold Delta table (Weeks 1-3) and reuses the Week 4 modules `src/ml/training_dataset.py`, `src/ml/features.py` and `src/ml/train.py`, driven by `config/ml.yaml`.

Then both are compared on implementation complexity, preprocessing complexity, training time and reproducibility.

**Conventions**
* Code cells starting with `# [A]` / `# [B]` contain the code of that approach. They are counted for the implementation-complexity comparison (blank lines, comments, `step(...)` and `tracker` lines are excluded).
* All other code cells only *measure* the two approaches (timers, checksums, comparisons) and are not part of either workflow.
* Approach A's data preparation follows the same rules as the platform (Week 1 validation rules, Gold join logic) so that the two datasets can be compared row by row.

**Prerequisites:** the original raw files in `data/raw/` (A), and Week 1 ingestion + integration (B). Outputs go to `data/ml/comparison/`, `data/delta/ml/comparison/`, `models/comparison/` and `reports/week4/comparison/`, so the main Week 4 artifacts are not overwritten.

In [1]:
import os, sys, json, glob, math

import pandas as pd

def find_project_root(marker="config/datasets.yaml"):
    path = os.getcwd()
    while path != os.path.dirname(path):
        if os.path.exists(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)
    raise FileNotFoundError(f"Could not find project root (looking for {marker})")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from pyspark.sql import Window
from pyspark.sql import functions as F
from src.common.spark_session import get_spark_session

# both approaches run in the same Spark session with the same settings
spark = get_spark_session("ml-approach-comparison")
# warm-up job so that neither approach pays for JVM / code generation start-up
spark.range(5_000_000).selectExpr("sum(id)").collect()

[Row(sum(id)=12499997500000)]

### Measurement helpers
Not part of either workflow: stage timers (wall time, peak memory and Spark work per stage), order-independent checksums and a log of Approach A's preprocessing steps.

In [2]:
from src.ml.run_metrics import RunTracker, code_stats, config_lines, dataframe_checksum

NOTEBOOK_PATH = "notebooks/week4/task4_approach_comparison.ipynb"
REPORT_PATH = "reports/week4/comparison/approach_comparison.json"
REPRO_CHECK = True        # build each training dataset a second time and compare checksums

KEYS = ["pickup_location_id", "hour_ts"]

# every preprocessing step Approach A has to implement itself
A_STEPS = []
def step(category, description):
    A_STEPS.append({"category": category, "step": description})


def normalized(df, columns):
    # same representation for both approaches: strings, NaN -> NULL, doubles rounded to 6 decimals
    types = dict(df.dtypes)
    out = []
    for c in columns:
        x = F.col(c)
        if types[c] in ("double", "float"):
            x = F.round(F.nanvl(x, F.lit(None).cast("double")), 6)
        out.append(x.cast("string").alias(c))
    return df.select(*out)


def dataset_checksum(df, columns):
    return dataframe_checksum(normalized(df, columns), columns)


def stage_seconds(stages, name):
    return stages.get(name, {}).get("seconds")

---
## Approach A - build the training dataset from the raw files

### A.0 Settings
There is no shared configuration: every path, rule and parameter is written into the notebook. The values are the ones in `config/ml.yaml`, so both approaches build the same dataset (checked in the comparison).

In [3]:
# [A] all settings are hard-coded in the notebook
RAW = "data/raw"
TRIP_FILES = sorted(glob.glob(f"{RAW}/tripdata/yellow_tripdata_2024-*.parquet"))   # original release only
WEATHER_FILE = f"{RAW}/weather/weather.csv"
AIR_FILE = f"{RAW}/air_quality/air_quality.csv"
ZONE_FILE = f"{RAW}/taxi_zone_lookup/taxi_zone_lookup.csv"
A_TRAINING_PATH = "data/ml/comparison/approach_a/training_dataset"

LOCAL_TZ = "America/New_York"                  # taxi timestamps are local wall-clock time
START, END = "2024-01-01 05:00:00", "2024-04-01 04:00:00"   # UTC
EXCLUDE_ZONES = [264, 265]                     # "Unknown" / "N/A" zones
MIN_ZONE_TRIPS = 100
LAGS, ROLLING = [1, 24, 168], 24
TRAIN_RATIO, VALIDATION_RATIO = 0.70, 0.15
GBT_PARAMS = dict(maxIter=50, maxDepth=6, stepSize=0.1, subsamplingRate=0.8, seed=42)

# county names in the air-quality file -> taxi boroughs
COUNTY_TO_BOROUGH = {"Bronx": "Bronx", "Kings": "Brooklyn", "New York": "Manhattan",
                     "Queens": "Queens", "Richmond": "Staten Island"}

WEATHER_FEATURES = ["weather_temperature", "weather_relative_humidity", "weather_precipitation",
                    "weather_wind_speed", "weather_pressure", "weather_condition_code"]
AIR_FEATURES = ["air_quality_pm25", "air_quality_station_count"]
HISTORY = [*[f"demand_lag_{h}h" for h in LAGS], f"demand_rolling_mean_{ROLLING}h"]
COLUMNS = ["pickup_location_id", "hour_ts", "pickup_zone", "pickup_borough", *WEATHER_FEATURES,
           *AIR_FEATURES, *HISTORY, "trip_count", "split"]

### A.1 Load the raw files
Each format needs its own reader. The raw column names differ between the datasets (`PULocationID`, `temp`, `Sample Measurement`, `LocationID`, ...).

In [4]:
# [A] 1. load the raw files and check the columns the later steps depend on
TRIP_COLUMNS = ["tpep_pickup_datetime", "passenger_count", "trip_distance", "PULocationID"]
WEATHER_COLUMNS = ["year", "month", "day", "hour", "temp", "rhum", "prcp", "wspd", "pres", "coco"]
AIR_COLUMNS = ["State Code", "County Code", "Site Num", "Parameter Code", "POC", "Latitude",
               "Longitude", "Date GMT", "Time GMT", "Sample Measurement", "County Name"]
ZONE_COLUMNS = ["LocationID", "Borough", "Zone"]


def require(df, columns, name):
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise ValueError(f"{name}: missing columns {missing}")
    return df


def load_raw():
    csv = spark.read.option("header", True)
    trips = spark.read.parquet(*TRIP_FILES).withColumn("_file", F.input_file_name())
    return {
        "trips": require(trips, TRIP_COLUMNS, "taxi trips"),
        "weather": require(csv.csv(WEATHER_FILE), WEATHER_COLUMNS, "weather"),
        "air": require(csv.csv(AIR_FILE), AIR_COLUMNS, "air quality"),
        "zones": require(csv.csv(ZONE_FILE), ZONE_COLUMNS, "taxi zones"),
    }

step("load", "read 3 Parquet files and 3 CSV files with format-specific readers")
step("schema", "check that the raw columns used later exist in each file")

### A.2 Clean and validate the context datasets
Type casts, timestamp construction, range checks and duplicate keys, following the platform's Week 1 rules (`config/datasets.yaml`).

In [5]:
# [A] 2. clean the weather, air-quality and zone files

def cast_failed(raw, typed):
    return raw.isNotNull() & typed.isNull()


def not_finite(column):
    return F.coalesce(F.isnan(column) | (F.abs(column) == float("inf")), F.lit(False))


def drop_duplicated_keys(df, keys):
    # a key that occurs more than once is ambiguous: every copy is rejected (platform rule)
    counts = F.count(F.lit(1)).over(Window.partitionBy(*keys))
    return df.withColumn("_n", counts).filter("_n = 1").drop("_n")


def clean_weather(raw):
    ints = ["year", "month", "day", "hour", "coco"]
    doubles = {"temp": "weather_temperature", "rhum": "weather_relative_humidity",
               "prcp": "weather_precipitation", "wspd": "weather_wind_speed", "pres": "weather_pressure"}
    df = raw.select(*[F.col(c).alias(f"raw_{c}") for c in [*ints, *doubles]],
                    *[F.col(c).cast("int").alias(c) for c in ints],
                    *[F.col(c).cast("double").alias(name) for c, name in doubles.items()])

    # Meteostat hours are UTC: build the hour timestamp from its parts
    df = df.withColumn("hour_ts", F.to_timestamp(
        F.format_string("%04d-%02d-%02d %02d:00:00", "year", "month", "day", "hour")))

    valid = F.col("hour_ts").isNotNull()
    for c in ints:
        valid &= ~cast_failed(F.col(f"raw_{c}"), F.col(c))
    for c, name in doubles.items():
        valid &= ~cast_failed(F.col(f"raw_{c}"), F.col(name)) & ~not_finite(F.col(name))
    # missing values are allowed, out-of-range values are not
    valid &= F.coalesce(F.col("weather_relative_humidity").between(0, 100), F.lit(True))
    valid &= F.coalesce(F.col("weather_precipitation") >= 0, F.lit(True))
    valid &= F.coalesce(F.col("weather_wind_speed") >= 0, F.lit(True))

    df = drop_duplicated_keys(df.filter(valid), ["hour_ts"])
    return df.select("hour_ts", *list(doubles.values()), F.col("coco").alias("weather_condition_code"))


def clean_air_quality(raw):
    df = raw.filter(F.col("State Code") == "36").select(        # only New York State can match a borough
        F.col("State Code").alias("state"), F.col("County Code").alias("county_code"),
        F.col("Site Num").alias("site"), F.col("Parameter Code").alias("parameter"),
        F.col("POC").alias("raw_poc"), F.col("POC").cast("int").alias("poc"),
        F.col("Latitude").cast("double").alias("latitude"),
        F.col("Longitude").cast("double").alias("longitude"),
        F.col("Sample Measurement").cast("double").alias("pm25"),
        F.col("County Name").alias("county"),
        F.to_timestamp(F.concat_ws(" ", "Date GMT", "Time GMT"), "yyyy-MM-dd HH:mm").alias("event_ts"),
    )
    keys = ["state", "county_code", "site", "parameter", "poc", "event_ts"]
    valid = F.lit(True)
    for k in keys:
        valid &= F.col(k).isNotNull()
    valid &= ~cast_failed(F.col("raw_poc"), F.col("poc"))
    valid &= F.col("latitude").between(-90, 90) & F.col("longitude").between(-180, 180)
    valid &= (F.col("pm25") >= -5) & ~not_finite(F.col("pm25"))     # small negative values = sensor noise

    borough = F.create_map(*[F.lit(x) for pair in COUNTY_TO_BOROUGH.items() for x in pair])
    df = (drop_duplicated_keys(df.filter(valid), keys)
          .withColumn("pickup_borough", borough[F.col("county")])
          .filter(F.col("pickup_borough").isNotNull()))

    # several stations per borough: average them per (borough, hour)
    return df.groupBy("pickup_borough", F.date_trunc("hour", "event_ts").alias("hour_ts")).agg(
        F.avg("pm25").alias("air_quality_pm25"),
        F.count(F.lit(1)).alias("air_quality_station_count"))


def clean_zones(raw):
    df = raw.select(F.col("LocationID").cast("int").alias("pickup_location_id"),
                    F.col("Zone").alias("pickup_zone"), F.col("Borough").alias("pickup_borough"))
    return drop_duplicated_keys(df.filter(F.col("pickup_location_id") >= 1), ["pickup_location_id"])

step("types", "weather: cast 5 integer and 5 decimal columns, reject rows whose values cannot be cast")
step("timestamp", "weather: build a UTC hour timestamp from year/month/day/hour")
step("validation", "weather: range checks (humidity 0-100, precipitation >= 0, wind >= 0), reject NaN/inf")
step("deduplication", "weather: reject hours that occur more than once")
step("filter", "air quality: keep New York State only (state code 36)")
step("types", "air quality: cast POC, latitude, longitude and PM2.5")
step("timestamp", "air quality: build a UTC timestamp from 'Date GMT' + 'Time GMT'")
step("validation", "air quality: non-null key, latitude/longitude range, PM2.5 >= -5, reject NaN/inf")
step("deduplication", "air quality: reject duplicated (site, parameter, POC, timestamp) keys")
step("mapping", "air quality: map county names to taxi boroughs")
step("aggregation", "air quality: average all stations per borough and hour")
step("types", "zones: cast the location id, reject invalid ids")
step("deduplication", "zones: reject duplicated location ids")

### A.3 Clean the taxi trips
The largest dataset: exact duplicate rows, local-time to UTC conversion, validity rules, and the zone/time filters of the prediction problem.

In [6]:
# [A] 3. clean the taxi trips

def clean_trips(raw):
    source_columns = [c for c in raw.columns if c != "_file"]
    # an exact copy of a row inside one file is ambiguous: every copy is rejected (platform rule);
    # the same row in two different files is kept once
    df = (raw.groupBy("_file", *source_columns).count().filter("count = 1")
             .dropDuplicates(source_columns))

    # pickup time is New York wall-clock time -> UTC
    pickup = F.to_utc_timestamp(F.to_timestamp(F.col("tpep_pickup_datetime").cast("string")), LOCAL_TZ)
    df = df.select(F.col("PULocationID").cast("int").alias("pickup_location_id"),
                   F.col("passenger_count").cast("int").alias("passenger_count"),
                   F.col("trip_distance").cast("double").alias("trip_distance"),
                   pickup.alias("pickup_ts"))

    valid = F.col("pickup_ts").isNotNull()
    valid &= (F.col("trip_distance") >= 0) & ~not_finite(F.col("trip_distance"))
    valid &= F.coalesce(F.col("passenger_count") >= 0, F.lit(True))
    valid &= (F.year("pickup_ts") == 2024) & F.month("pickup_ts").between(1, 4)

    in_window = ((F.col("pickup_ts") >= F.lit(START).cast("timestamp"))
                 & (F.col("pickup_ts") < F.lit(END).cast("timestamp")))
    known_zone = F.col("pickup_location_id").isNotNull() & ~F.col("pickup_location_id").isin(EXCLUDE_ZONES)

    return (df.filter(valid & in_window & known_zone)
              .select("pickup_location_id", F.date_trunc("hour", "pickup_ts").alias("hour_ts")))

step("deduplication", "trips: reject exact duplicate rows within a file, keep one copy across files")
step("timestamp", "trips: convert New York local pickup time to UTC (DST aware)")
step("validation", "trips: trip_distance >= 0 and finite, passenger_count >= 0 or missing, year/month in range")
step("filter", "trips: time window of the prediction problem, drop unknown zones 264/265")

### A.4 Integrate and build the zone x hour dataset
Enrich every trip with its zone, the weather and the borough's air quality at pickup time (what the Gold table does), then aggregate to zones and hours and add the history features and the chronological split.

In [7]:
# [A] 4. integrate the datasets, then build the training dataset

def integrate(raw):
    trips = clean_trips(raw["trips"])
    zones = clean_zones(raw["zones"])
    weather = clean_weather(raw["weather"])
    air = clean_air_quality(raw["air"])
    # one row per trip: pickup zone + weather (hour) + air quality (borough, hour)
    return (trips.join(F.broadcast(zones), "pickup_location_id", "left")
                 .join(F.broadcast(weather), "hour_ts", "left")
                 .join(F.broadcast(air), ["pickup_borough", "hour_ts"], "left"))


def build_training_dataset(enriched, path):
    demand = enriched.groupBy(*KEYS).agg(F.count(F.lit(1)).alias("trip_count")).persist()
    bounds = demand.agg(F.min(F.col("hour_ts").cast("long")).alias("start"),
                        F.max(F.col("hour_ts").cast("long")).alias("end")).first()
    start, end = int(bounds["start"]), int(bounds["end"])

    # chronological split boundaries
    total_hours = (end - start) // 3600 + 1
    train_end = start + int(total_hours * TRAIN_RATIO) * 3600
    validation_end = train_end + int(total_hours * VALIDATION_RATIO) * 3600

    # zones with enough trips in the training period; every zone gets every hour (0 = no trips)
    active = (demand.filter(F.col("hour_ts").cast("long") < train_end).groupBy("pickup_location_id")
                    .agg(F.sum("trip_count").alias("n")).filter(F.col("n") >= MIN_ZONE_TRIPS)
                    .select("pickup_location_id"))
    hours = spark.range(start, end + 1, 3600).select(F.timestamp_seconds("id").alias("hour_ts"))
    data = (active.crossJoin(hours).join(demand, KEYS, "left")
                  .withColumn("trip_count", F.coalesce("trip_count", F.lit(0)).cast("int")))

    # context: zone attributes per zone, weather per hour, air quality per borough and hour
    first = lambda cols: [F.first(c, ignorenulls=True).alias(c) for c in cols]
    zone_attrs = enriched.groupBy("pickup_location_id").agg(*first(["pickup_zone", "pickup_borough"]))
    hourly_weather = enriched.groupBy("hour_ts").agg(*first(WEATHER_FEATURES))
    hourly_air = (enriched.filter(F.col("pickup_borough").isNotNull())
                          .groupBy("pickup_borough", "hour_ts").agg(*first(AIR_FEATURES)))
    data = (data.join(zone_attrs, "pickup_location_id", "left")
                .join(hourly_weather, "hour_ts", "left")
                .join(hourly_air, ["pickup_borough", "hour_ts"], "left"))

    # demand history of the same zone, past hours only
    ordered = Window.partitionBy("pickup_location_id").orderBy("hour_ts")
    for h in LAGS:
        data = data.withColumn(f"demand_lag_{h}h", F.lag("trip_count", h).over(ordered))
    past = ordered.rowsBetween(-ROLLING, -1)
    data = data.withColumn(f"demand_rolling_mean_{ROLLING}h",
                           F.when(F.count("trip_count").over(past) == ROLLING, F.avg("trip_count").over(past)))
    data = data.dropna(subset=HISTORY)

    epoch = F.col("hour_ts").cast("long")
    data = data.withColumn("split", F.when(epoch < train_end, "train")
                                     .when(epoch < validation_end, "validation").otherwise("test"))
    data.select(*COLUMNS).write.mode("overwrite").partitionBy("split").parquet(path)
    demand.unpersist()
    return {"start": start, "end": end, "train_end": train_end, "validation_end": validation_end}

step("join", "enrich trips with pickup zone and borough (zone lookup)")
step("join", "enrich trips with hourly weather")
step("join", "enrich trips with air quality of the pickup borough and hour")
step("aggregation", "count trips per zone and hour (the target)")
step("split", "compute chronological train/validation/test boundaries")
step("filter", "keep zones with >= 100 trips in the training period")
step("densify", "zone x hour grid, hours without trips get demand 0")
step("join", "attach zone attributes, hourly weather and borough-hourly air quality to the grid")
step("history", "lagged demand 1h / 24h / 168h and 24h rolling mean, drop rows without full history")
step("output", "write the training dataset partitioned by split")

In [8]:
# [A] run the data preparation
tracker_a = RunTracker(spark)
with tracker_a.stage("clean_and_integrate"):
    raw = load_raw()
    enriched = integrate(raw).persist()
    enriched_rows = enriched.count()
with tracker_a.stage("build_training_dataset"):
    info_a = build_training_dataset(enriched, A_TRAINING_PATH)
enriched.unpersist()
print(f"valid trips in the problem window: {enriched_rows:,}")

valid trips in the problem window: 9,517,995


In [9]:
spark.read.parquet(A_TRAINING_PATH).groupBy("split").agg(
    F.count("*").alias("rows"), F.countDistinct("pickup_location_id").alias("zones"),
    F.round(F.avg("trip_count"), 4).alias("mean_target"),
    F.min("hour_ts").cast("string").alias("first_hour"), F.max("hour_ts").cast("string").alias("last_hour"),
).orderBy("first_hour").toPandas()

,split,rows,zones,mean_target,first_hour,last_hour
0,train,301920,222,18.9753,2024-01-08 05:00:00,2024-03-04 20:00:00
1,validation,72594,222,22.1587,2024-03-04 21:00:00,2024-03-18 11:00:00
2,test,72816,222,21.5969,2024-03-18 12:00:00,2024-04-01 03:00:00


### A.5 Feature engineering
Written by hand for this one dataset: the column names, the imputation rules and the feature order are fixed in code.

In [10]:
# [A] 5. feature engineering
import holidays
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, OneHotEncoder, StandardScaler, StringIndexer, VectorAssembler

HOLIDAYS = sorted(d.isoformat() for d in holidays.country_holidays("US", subdiv="NY", years=[2024]))
NUMERIC = ["weather_temperature", "weather_relative_humidity", "weather_precipitation",
           "weather_wind_speed", "weather_pressure", "air_quality_pm25", *HISTORY]
CATEGORICAL = ["pickup_location_id", "pickup_borough", "weather_condition_code"]
NUMERIC_BLOCK = ["hour_of_day_sin", "hour_of_day_cos", "day_of_week_sin", "day_of_week_cos",
                 "month", "is_weekend", "is_holiday", *[f"{c}_imp" for c in NUMERIC],
                 "weather_precipitation_missing", "air_quality_pm25_missing"]


def add_features(df):
    local = F.from_utc_timestamp(F.col("hour_ts").cast("timestamp"), LOCAL_TZ)
    hour, weekday = F.hour(local), (F.dayofweek(local) + 5) % 7          # Monday = 0
    df = (df.withColumn("hour_of_day_sin", F.sin(F.lit(2 * math.pi) * hour / 24.0))
            .withColumn("hour_of_day_cos", F.cos(F.lit(2 * math.pi) * hour / 24.0))
            .withColumn("day_of_week_sin", F.sin(F.lit(2 * math.pi) * weekday / 7.0))
            .withColumn("day_of_week_cos", F.cos(F.lit(2 * math.pi) * weekday / 7.0))
            .withColumn("month", F.month(local))
            .withColumn("is_weekend", F.dayofweek(local).isin(1, 7).cast("int"))
            .withColumn("is_holiday", F.to_date(local).cast("string").isin(HOLIDAYS).cast("int")))
    for c in NUMERIC:
        df = df.withColumn(f"{c}_clean", F.nanvl(F.col(c).cast("double"), F.lit(None).cast("double")))
    # missing-value flags, then rule-based filling: no precipitation recorded = 0,
    # no PM2.5 monitor in the borough = mean of the other boroughs in the same hour
    df = (df.withColumn("weather_precipitation_missing", F.col("weather_precipitation_clean").isNull().cast("double"))
            .withColumn("air_quality_pm25_missing", F.col("air_quality_pm25_clean").isNull().cast("double"))
            .withColumn("weather_precipitation_clean", F.coalesce("weather_precipitation_clean", F.lit(0.0)))
            .withColumn("air_quality_pm25_clean", F.coalesce(
                "air_quality_pm25_clean", F.avg("air_quality_pm25_clean").over(Window.partitionBy("hour_ts")))))
    return df.withColumn("label", F.col("trip_count").cast("double"))


# stages fitted on the training split: median fill, one-hot encoding, scaling, final vector
feature_stages = [
    Imputer(strategy="median", inputCols=[f"{c}_clean" for c in NUMERIC], outputCols=[f"{c}_imp" for c in NUMERIC]),
    StringIndexer(inputCols=CATEGORICAL, outputCols=[f"{c}_idx" for c in CATEGORICAL], handleInvalid="keep"),
    OneHotEncoder(inputCols=[f"{c}_idx" for c in CATEGORICAL], outputCols=[f"{c}_ohe" for c in CATEGORICAL],
                  handleInvalid="keep"),
    VectorAssembler(inputCols=NUMERIC_BLOCK, outputCol="numeric_features"),
    StandardScaler(inputCol="numeric_features", outputCol="numeric_scaled", withMean=True, withStd=True),
    VectorAssembler(inputCols=["numeric_scaled", *[f"{c}_ohe" for c in CATEGORICAL]], outputCol="features"),
]

step("temporal", "hour, weekday, month, weekend and holiday in New York local time; sin/cos for hour and weekday")
step("missing", "treat NaN as missing, add missing flags for precipitation and PM2.5")
step("missing", "fill precipitation with 0 and PM2.5 with the hourly mean of the other boroughs")
step("missing", "median fill (training split) for anything still missing")
step("encoding", "one-hot encode zone, borough and weather condition, with a slot for unseen values")
step("scaling", "standardise the numeric block (training split statistics)")
step("output", "assemble the final feature vector and the label")

### A.6 Train and evaluate
Same estimator and parameters as Approach B. The rows get the same fixed partition layout before training, because GBT row subsampling depends on how rows are split into partitions.

In [11]:
# [A] 6. train and evaluate
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.regression import GBTRegressor

with tracker_a.stage("load_train"):
    data_a = (spark.read.parquet(A_TRAINING_PATH)
                   .repartition(8, *KEYS).sortWithinPartitions("hour_ts", "pickup_location_id"))
    train_a = data_a.filter(F.col("split") == "train").cache()
    train_rows_a = train_a.count()

with tracker_a.stage("feature_engineering"):
    features_a = Pipeline(stages=feature_stages).fit(add_features(train_a))
    train_features_a = features_a.transform(add_features(train_a)).cache()
    train_features_a.count()

with tracker_a.stage("model_fit"):
    gbt_a = GBTRegressor(featuresCol="features", labelCol="label", **GBT_PARAMS).fit(train_features_a)
train_features_a.unpersist()


def predict_a(df):
    return gbt_a.transform(features_a.transform(add_features(df)))


def regression_metrics(pred):
    pred = pred.withColumn("prediction", F.greatest(F.col("prediction"), F.lit(0.0))).cache()   # demand >= 0
    out = {m: round(RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName=m).evaluate(pred), 4)
           for m in ("rmse", "mae", "r2")}
    totals = pred.agg(F.sum(F.abs(F.col("label") - F.col("prediction"))).alias("error"),
                      F.sum("label").alias("demand")).first()
    out["wape"] = round(totals["error"] / totals["demand"], 4)
    out["rows"] = pred.count()
    pred.unpersist()
    return out


with tracker_a.stage("evaluation"):
    metrics_a = {s: regression_metrics(predict_a(data_a.filter(F.col("split") == s)))
                 for s in ("train", "validation", "test")}
train_a.unpersist()
pd.DataFrame(metrics_a).T

,rmse,mae,r2,wape,rows
train,10.0671,3.4878,0.9659,0.1838,301920.0
validation,14.5698,4.7480,0.9441,0.2143,72594.0
test,13.7952,4.5286,0.9457,0.2097,72816.0


---
## Approach B - build the training dataset from the platform

The comparison writes B's outputs to separate locations so the main Week 4 dataset, model and reports are not overwritten. That redirect is set up in the measurement cell below and is not part of B's workflow.

In [12]:
from dataclasses import replace
from src.ml.config import load_ml_config

base_cfg = load_ml_config()
cfg = replace(base_cfg,
              training_dataset="data/delta/ml/comparison/approach_b/training_dataset",
              manifest="reports/week4/comparison/approach_b_manifest.json",
              model=replace(base_cfg.model, directory="models/comparison/approach_b",
                            metrics_report="reports/week4/comparison/approach_b_metrics.json"))
spark.catalog.clearCache()
tracker_b = RunTracker(spark)

In [13]:
# [B] the whole workflow: Gold table (Weeks 1-3) -> training dataset -> feature pipeline + model
from src.ml.training_dataset import generate_training_dataset
from src.ml.train import train_model

with tracker_b.stage("build_training_dataset"):
    manifest_b = generate_training_dataset(spark, cfg)
summary_b = train_model(spark, cfg)

In [14]:
pd.DataFrame({s: summary_b["metrics"][s]["model"] for s in ("train", "validation", "test")}).T

,rmse,mae,r2,wape,rows
train,9.9858,3.4867,0.9664,0.1837,301920.0
validation,14.5624,4.7780,0.9441,0.2156,72594.0
test,13.8205,4.5425,0.9455,0.2103,72816.0


### One-off cost of the platform
B starts from tables the platform has already built. The time to ingest the original files into Silver is read from the platform's ingestion metadata (most recent full load of each original file). It is paid once and reused by every later analysis and ML task.

In [15]:
from deltalake import DeltaTable

runs = DeltaTable("data/delta/metadata/ingestion_runs").to_pandas()
originals = runs[(runs["status"] == "SUCCESS") & (runs["records_valid_silver"] > 0)
                 & ~runs["source_file"].str.contains("update|_validation")]
latest_full_load = originals.sort_values("started_at").groupby("source_file").last()
platform_ingestion = latest_full_load.groupby("dataset_name")["execution_time_seconds"].sum().round(2)

pipeline_runs = DeltaTable("data/delta/metadata/pipeline_runs").to_pandas()
full_gold = pipeline_runs[(pipeline_runs["pipeline_name"] == "gold_build")
                          & (pipeline_runs["records_inserted"] > 1_000_000)]
gold_build_seconds = float(full_gold["execution_time_seconds"].iloc[-1]) if len(full_gold) else None

print("Gold full build:", f"{gold_build_seconds:.1f}s" if gold_build_seconds else "not logged (only incremental builds are recorded)")
platform_ingestion.to_frame("silver_ingestion_seconds")

Gold full build: not logged (only incremental builds are recorded)


,silver_ingestion_seconds
dataset_name,
air_quality,40.40
taxi_zone_lookup,3.32
tripdata,106.83
weather,1.92


---
## Comparison

### 1. Do both approaches build the same training dataset?
The two datasets are joined on (zone, hour) and compared column by column, after NaN is mapped to NULL and doubles are rounded to 6 decimals.

In [16]:
from src.ml.training_dataset import load_training_dataset

ds_a = spark.read.parquet(A_TRAINING_PATH)
ds_b = load_training_dataset(spark, cfg)
assert set(ds_a.columns) == set(ds_b.columns), set(ds_a.columns) ^ set(ds_b.columns)

na, nb = normalized(ds_a, COLUMNS).alias("a"), normalized(ds_b, COLUMNS).alias("b")
joined = na.join(nb, [F.col(f"a.{k}") == F.col(f"b.{k}") for k in KEYS], "full_outer")
values = [c for c in COLUMNS if c not in KEYS]
diff = joined.agg(
    F.sum(F.col("b.hour_ts").isNull().cast("int")).alias("rows_only_in_A"),
    F.sum(F.col("a.hour_ts").isNull().cast("int")).alias("rows_only_in_B"),
    *[F.sum((F.col("a.hour_ts").isNotNull() & F.col("b.hour_ts").isNotNull()
             & ~F.col(f"a.{c}").eqNullSafe(F.col(f"b.{c}"))).cast("int")).alias(c) for c in values],
).first().asDict()

checksum_a, checksum_b = dataset_checksum(ds_a, COLUMNS), dataset_checksum(ds_b, COLUMNS)
same_dataset = {
    "rows_A": ds_a.count(), "rows_B": ds_b.count(),
    "rows_only_in_A": diff.pop("rows_only_in_A"), "rows_only_in_B": diff.pop("rows_only_in_B"),
    "mismatched_values_per_column": diff,
    "split_boundaries_equal": (info_a["train_end"], info_a["validation_end"]) == tuple(
        int(pd.Timestamp(manifest_b["split_boundaries_utc"][k], tz="UTC").timestamp())
        for k in ("train_end", "validation_end")),
    "checksum_A": checksum_a, "checksum_B": checksum_b, "identical": checksum_a == checksum_b,
}
print(json.dumps({k: v for k, v in same_dataset.items() if k != "mismatched_values_per_column"}, indent=2))
pd.Series(same_dataset["mismatched_values_per_column"], name="mismatched rows").to_frame().T

{
  "rows_A": 447330,
  "rows_B": 447330,
  "rows_only_in_A": 0,
  "rows_only_in_B": 0,
  "split_boundaries_equal": true,
  "checksum_A": 6214592431739770209,
  "checksum_B": 6214592431739770209,
  "identical": true
}


,pickup_zone,pickup_borough,weather_temperature,weather_relative_humidity,weather_precipitation,weather_wind_speed,weather_pressure,weather_condition_code,air_quality_pm25,air_quality_station_count,demand_lag_1h,demand_lag_24h,demand_lag_168h,demand_rolling_mean_24h,trip_count,split
mismatched rows,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


### 2. Do both approaches produce the same model?

In [17]:
test_pred_a = predict_a(data_a.filter(F.col("split") == "test"))
pred_cols = ["pickup_location_id", "hour_ts", "prediction"]
predictions_checksum_a = dataframe_checksum(
    test_pred_a.select(*KEYS, F.round("prediction", 4).alias("prediction")), pred_cols)
predictions_checksum_b = summary_b["reproducibility"]["predictions_checksum"]

rows = []
for s in ("train", "validation", "test"):
    for approach, m in (("A (raw)", metrics_a[s]), ("B (platform)", summary_b["metrics"][s]["model"])):
        rows.append({"split": s, "approach": approach, **m})
print("test predictions identical:", predictions_checksum_a == predictions_checksum_b)
pd.DataFrame(rows)

test predictions identical: False


,split,approach,rmse,mae,r2,wape,rows
0,train,A (raw),10.0671,3.4878,0.9659,0.1838,301920
1,train,B (platform),9.9858,3.4867,0.9664,0.1837,301920
2,validation,A (raw),14.5698,4.7480,0.9441,0.2143,72594
3,validation,B (platform),14.5624,4.7780,0.9441,0.2156,72594
4,test,A (raw),13.7952,4.5286,0.9457,0.2097,72816
5,test,B (platform),13.8205,4.5425,0.9455,0.2103,72816


### 3. Implementation complexity
Counted from the `# [A]` / `# [B]` cells of this notebook. "Raw columns referenced" counts how many source column names of the original files the code has to know.

In [18]:
with open(NOTEBOOK_PATH) as f:
    notebook = json.load(f)

def approach_lines(tag):
    lines = []
    for cell in notebook["cells"]:
        source = "".join(cell["source"])
        if cell["cell_type"] == "code" and source.startswith(tag):
            lines += [l for l in source.splitlines()
                      if l.strip() and not l.strip().startswith("#")
                      and not l.strip().startswith("step(") and "tracker" not in l]
    return lines

lines_a, lines_b = approach_lines("# [A]"), approach_lines("# [B]")
raw_columns = {c for df in raw.values() for c in df.columns} - {"_file"}
raw_used = lambda lines: sorted(c for c in raw_columns if f'"{c}"' in "\n".join(lines))

ML_CODE = ["src/ml"]
PLATFORM_CODE = ["src/common", "src/ingestion", "src/transformation", "src/integration"]
ml_stats, platform_stats = code_stats(ML_CODE), code_stats(PLATFORM_CODE)

complexity = pd.DataFrame({
    "A (raw)": {
        "lines of code written for this task": len(lines_a),
        "functions written for this task": sum(l.strip().startswith("def ") for l in lines_a),
        "raw source columns referenced by name": len(raw_used(lines_a)),
        "settings hard-coded in code": "all",
        "reused, already-tested code (lines)": 0,
    },
    "B (platform)": {
        "lines of code written for this task": len(lines_b),
        "functions written for this task": sum(l.strip().startswith("def ") for l in lines_b),
        "raw source columns referenced by name": len(raw_used(lines_b)),
        "settings hard-coded in code": f"none ({config_lines('config/ml.yaml')} lines of config/ml.yaml)",
        "reused, already-tested code (lines)": ml_stats["lines_of_code"] + platform_stats["lines_of_code"],
    },
})
complexity

,A (raw),B (platform)
lines of code written for this task,226,4
functions written for this task,14,0
raw source columns referenced by name,28,0
settings hard-coded in code,all,none (81 lines of config/ml.yaml)
"reused, already-tested code (lines)",0,1743


In [19]:
# Approach A duplicates settings that live once in config/ml.yaml for B; a change in one place
# silently diverges from the other
duplicated = {
    "data_start": (START, base_cfg.data_start), "data_end": (END, base_cfg.data_end),
    "local_timezone": (LOCAL_TZ, base_cfg.local_timezone),
    "exclude_zone_ids": (EXCLUDE_ZONES, base_cfg.exclude_zone_ids),
    "min_zone_trips": (MIN_ZONE_TRIPS, base_cfg.min_zone_trips),
    "lag_hours": (LAGS, base_cfg.lag_hours), "rolling_mean_hours": ([ROLLING], base_cfg.rolling_mean_hours),
    "split ratios": ((TRAIN_RATIO, VALIDATION_RATIO), (base_cfg.split.train_ratio, base_cfg.split.validation_ratio)),
    "model params": (GBT_PARAMS, base_cfg.model.params),
}
pd.DataFrame([{"setting": k, "A (hard-coded)": str(a), "B (config/ml.yaml)": str(b), "equal": a == b}
              for k, (a, b) in duplicated.items()])

,setting,A (hard-coded),B (config/ml.yaml),equal
0,data_start,2024-01-01 05:00:00,2024-01-01 05:00:00,True
1,data_end,2024-04-01 04:00:00,2024-04-01 04:00:00,True
2,local_timezone,America/New_York,America/New_York,True
3,exclude_zone_ids,"[264, 265]","[264, 265]",True
4,min_zone_trips,100,100,True
5,lag_hours,"[1, 24, 168]","[1, 24, 168]",True
6,rolling_mean_hours,[24],[24],True
7,split ratios,"(0.7, 0.15)","(0.7, 0.15)",True
8,model params,"{'maxIter': 50, 'maxDepth': 6, 'stepSize': 0.1...","{'maxIter': 50, 'maxDepth': 6, 'stepSize': 0.1...",True


### 4. Preprocessing complexity
The preprocessing steps Approach A has to implement itself, grouped by type. In Approach B the ML code contains none of the loading, cleaning, validation or integration steps: they are config-driven rules of the Weeks 1-3 platform (counted from `config/datasets.yaml`). B's ML code only aggregates, adds history features and engineers features, from settings in `config/ml.yaml`.

In [20]:
import yaml

steps_a = pd.DataFrame(A_STEPS)
by_category = steps_a.groupby("category", sort=False).size().rename("A: steps written in the notebook")

with open("config/datasets.yaml") as f:
    platform_cfg = yaml.safe_load(f)["datasets"]
platform_rules = pd.DataFrame({
    name: {"column renames": len(d.get("column_mapping", {})), "type rules": len(d.get("dtype_overrides", {})),
           "validation rules": len(d.get("numeric_rules", {})), "primary-key columns": len(d.get("primary_key", []))}
    for name, d in platform_cfg.items()}).T

print(f"Approach A: {len(steps_a)} preprocessing steps written by hand")
display(by_category.to_frame())
print("Rules Approach B inherits from the platform (config/datasets.yaml):")
platform_rules

Approach A: 36 preprocessing steps written by hand


,A: steps written in the notebook
category,
load,1
schema,1
types,3
timestamp,3
validation,3
deduplication,4
filter,3
mapping,1
aggregation,2


Rules Approach B inherits from the platform (config/datasets.yaml):


,column renames,type rules,validation rules,primary-key columns
tripdata,6,11,4,1
weather,8,12,4,1
air_quality,15,9,4,6
taxi_zone_lookup,4,1,1,1


In [21]:
# Spark work of the data preparation (raw/Gold -> training dataset)
def spark_work(stages, names):
    total = {}
    for n in names:
        for k, v in (stages[n].get("spark") or {}).items():
            total[k] = round(total.get(k, 0) + v, 2)
    return total

prep_work = pd.DataFrame({
    "A (raw -> dataset)": spark_work(tracker_a.stages, ["clean_and_integrate", "build_training_dataset"]),
    "B (Gold -> dataset)": spark_work(tracker_b.stages, ["build_training_dataset"]),
})
prep_work = prep_work.loc[["stages", "tasks", "input_bytes", "shuffle_read_bytes", "executor_run_seconds"]]
prep_work.loc[["input_bytes", "shuffle_read_bytes"]] /= 1e6
prep_work.rename(index={"input_bytes": "input read (MB)", "shuffle_read_bytes": "shuffle read (MB)",
                        "executor_run_seconds": "executor CPU time (s)"}).round(1)

,A (raw -> dataset),B (Gold -> dataset)
stages,30.0,19.0
tasks,200.0,204.0
input read (MB),5124.3,405.9
shuffle read (MB),1110.5,7.2
executor CPU time (s),153.2,62.6


### 5. Training time
Wall time per stage in seconds. B's training stages come from `train_model`'s own tracker. B also profiles and saves a versioned model, which A does not do. B's evaluation also scores three naive baselines.

In [22]:
sa, sb = tracker_a.stages, {**tracker_b.stages, **summary_b["stages"]}
prep_a = stage_seconds(sa, "clean_and_integrate") + stage_seconds(sa, "build_training_dataset")
prep_b = stage_seconds(sb, "build_training_dataset")
train_stages = ["load_train", "feature_engineering", "model_fit", "evaluation"]

timing = pd.DataFrame({
    "A (raw)": {"data preparation -> training dataset": prep_a,
                **{s: stage_seconds(sa, s) for s in train_stages},
                "profiling + save model": None},
    "B (platform)": {"data preparation -> training dataset": prep_b,
                     **{s: stage_seconds(sb, s) for s in train_stages},
                     "profiling + save model": stage_seconds(sb, "profiling") + stage_seconds(sb, "save_model")},
})
timing.loc["total (excluding one-off platform cost)"] = timing.sum(numeric_only=True)
timing.loc["one-off: Silver ingestion of the original files"] = [None, float(platform_ingestion.sum())]
timing.loc["peak driver memory, data preparation (MB)"] = [
    max(sa["clean_and_integrate"]["peak_memory_mb"] or 0, sa["build_training_dataset"]["peak_memory_mb"] or 0),
    sb["build_training_dataset"]["peak_memory_mb"]]
timing.round(2)

,A (raw),B (platform)
data preparation -> training dataset,16.35,4.42
load_train,0.52,0.34
feature_engineering,3.07,2.08
model_fit,21.57,19.28
evaluation,3.51,4.14
profiling + save model,NaN,1.30
total (excluding one-off platform cost),45.02,31.56
one-off: Silver ingestion of the original files,NaN,152.47
"peak driver memory, data preparation (MB)",7441.10,7877.80


### 6. Reproducibility
Each training dataset is built a second time from the same inputs and the checksums are compared. Then we check what each approach records about how a result was produced.

In [23]:
rerun = {}
if REPRO_CHECK:
    with tracker_a.stage("rerun_build"):
        raw2 = load_raw()
        enriched2 = integrate(raw2).persist()
        build_training_dataset(enriched2, A_TRAINING_PATH + "_rerun")
        enriched2.unpersist()
    cfg_rerun = replace(cfg, training_dataset=cfg.training_dataset + "_rerun",
                        manifest=cfg.manifest.replace(".json", "_rerun.json"))
    with tracker_b.stage("rerun_build"):
        generate_training_dataset(spark, cfg_rerun)
    rerun = {
        "A": dataset_checksum(spark.read.parquet(A_TRAINING_PATH + "_rerun"), COLUMNS) == checksum_a,
        "B": dataset_checksum(load_training_dataset(spark, cfg_rerun), COLUMNS) == checksum_b,
        "A_seconds": tracker_a.stages["rerun_build"]["seconds"],
        "B_seconds": tracker_b.stages["rerun_build"]["seconds"],
    }

repro_b = summary_b["reproducibility"]
reproducibility = pd.DataFrame({
    "A (raw)": {
        "rebuild gives an identical dataset": rerun.get("A"),
        "rebuild time (s)": rerun.get("A_seconds"),
        "input version recorded": "no - raw files can change in place",
        "dataset manifest (splits, time range, row counts)": "no - only in notebook memory",
        "config / code version recorded": "no",
        "model saved and versioned": "no",
        "same seed + fixed row layout": "yes (set by hand)",
    },
    "B (platform)": {
        "rebuild gives an identical dataset": rerun.get("B"),
        "rebuild time (s)": rerun.get("B_seconds"),
        "input version recorded": f"Gold Delta version {manifest_b['source_gold_version']}",
        "dataset manifest (splits, time range, row counts)": cfg.manifest,
        "config / code version recorded": f"config {repro_b['config_hash']}, commit {str(repro_b['code_version'])[:8]}",
        "model saved and versioned": f"run {summary_b['run_id']}",
        "same seed + fixed row layout": "yes (built into train_model)",
    },
})
reproducibility

,A (raw),B (platform)
rebuild gives an identical dataset,True,True
rebuild time (s),13.77,3.82
input version recorded,no - raw files can change in place,Gold Delta version 3
"dataset manifest (splits, time range, row counts)",no - only in notebook memory,reports/week4/comparison/approach_b_manifest.json
config / code version recorded,no,"config 7152862e3de6d8c0, commit f013ab48"
model saved and versioned,no,run 20261004T162311655Z
same seed + fixed row layout,yes (set by hand),yes (built into train_model)


In [24]:
report = {
    "same_dataset": same_dataset,
    "same_model": {"metrics_A": metrics_a,
                   "metrics_B": {s: summary_b["metrics"][s]["model"] for s in ("train", "validation", "test")},
                   "predictions_checksum_A": predictions_checksum_a,
                   "predictions_checksum_B": predictions_checksum_b},
    "implementation": {"table": complexity.astype(str).to_dict(),
                       "raw_columns_used_A": raw_used(lines_a), "raw_columns_used_B": raw_used(lines_b)},
    "preprocessing": {"steps_A": A_STEPS, "steps_by_category_A": by_category.to_dict(),
                      "platform_rules_B": platform_rules.to_dict(orient="index"),
                      "spark_work": prep_work.to_dict()},
    "timing": {"table": timing.round(2).to_dict(), "stages_A": tracker_a.stages, "stages_B": sb,
               "platform_ingestion_seconds": platform_ingestion.to_dict(),
               "gold_full_build_seconds": gold_build_seconds},
    "reproducibility": reproducibility.astype(str).to_dict(),
}
os.makedirs(os.path.dirname(REPORT_PATH), exist_ok=True)
with open(REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2, default=str)
print("written", REPORT_PATH)

written reports/week4/comparison/approach_comparison.json


In [25]:
spark.stop()